### Step 3c: Plan for the dud games (schedule release)

Same model as Step 3a, with one new idea: **some games turn out to be duds.**

**The story in three steps:**
1. **July:** the station sells packages. It promises audience based on generic games (the baseline, 1,631.3 thousand households).
2. **August:** the NFL schedule comes out. Games with low-interest teams (Jets, Chargers, Texans, ...) lose 30% of their audience, or 40% if both teams are low-interest. The package's audience drops to **1,465.9**.
3. **The season:** Nielsen's new 1-minute rule raises the audience by 15%, 20%, or 25%, but from the smaller August number.

**What changed in the code:**
- **Data cell:** reads the real 27-game schedule and adds up the revised audience (1,465.9), with checks against the spreadsheet's totals.
- **Model, section 4 (one line):** actual audience = `revised_000s × (1 + increase)` instead of `baseline_000s × (1 + increase)`.
- **What stays the same:** guarantees and CPMs are still based on the July baseline, because that's what the customer was promised. Everything else is identical to Step 3a.

**What that does to delivery:**

| Nielsen increase | Probability | Delivered, July view (no duds) | Delivered, with duds |
| --- | ---: | ---: | ---: |
| +15% | 50% | 1,876.0 | 1,685.8 |
| +20% | 35% | 1,957.6 | 1,759.1 |
| +25% | 15% | 2,039.1 | 1,832.4 |

With the duds, **every guarantee above 0% now falls short in every scenario.** Medium (+15%) was risk-free in Step 3a. Now it misses by 44 to 190 thousand households.

**Assumption (a simplification to state in the report):** in reality the station sells in July, *before* it knows which games will be duds. Our model plans as if the station expects the dud cuts, using the August schedule as its best estimate. In other words, the station plans for dud risk. Reacting *after* the schedule comes out (for example, salespeople fixing shortfalls) is a natural extension.

In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
# ============================================================
# LOAD THE DATA
# ============================================================

# Ed's spreadsheet. Keep it in the same folder as this notebook.
# We only read from it; we never edit it.

DATA_FILE = "NFL Ad PKG MAIN.xlsx"


# ------------------------------------------------------------
# A. The package: 'Pre-Schedule Release', rows 13-17
# ------------------------------------------------------------

# One row per game type (Thanksgiving, noon regional, ...).
# This is what the station knows in July, before the schedule is out.
#
# col C = game type
# col E = number of games
# col F = rate per spot ($)
# col H = audience per game (000s = thousands of households)

df_games = pd.read_excel(
    DATA_FILE,
    sheet_name="Pre-Schedule Release",
    header=None,
    skiprows=12,
    nrows=5,
    usecols="C,E,F,H"
)

df_games.columns = ["game_type", "num_games", "rate", "audience_000s"]

package_price = (df_games["num_games"] * df_games["rate"]).sum()
baseline_000s = (df_games["num_games"] * df_games["audience_000s"]).sum()

# Check against the spreadsheet's own totals (row 18)
assert df_games["num_games"].sum() == 27
assert package_price == 108_000
assert abs(baseline_000s - 1631.3) < 0.001


# ------------------------------------------------------------
# B. Ratings scenarios: 'Schedule Release 1-Min Qualifer', T6:U8
# ------------------------------------------------------------

# Nielsen's new 1-minute rule raises ratings, but nobody knows
# by how much. Management gives 3 possible increases.
#
# col T = label, such as "15% Increase"
# col U = probability of that increase

df_scen = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=5,
    nrows=3,
    usecols="T:U"
)

df_scen.columns = ["label", "probability"]

# "15% Increase" -> 0.15
df_scen["increase"] = df_scen["label"].str.split("%").str[0].astype(float) / 100

# Check: probabilities add up to 100%
assert abs(df_scen["probability"].sum() - 1.0) < 0.000001


# ------------------------------------------------------------
# C. Prospects: 'Prospect Data Table', rows 4-33
# ------------------------------------------------------------

# col B = prospect ID
# col C = prospect name
# col D = salesperson        (not used until a later step)
# col E = close ratio        (not used until a later step)
# col F = goal CPM
# col G = makegood penalty multiplier (2, 3, or 4)

df_p = pd.read_excel(
    DATA_FILE,
    sheet_name="Prospect Data Table",
    skiprows=2,
    nrows=30,
    usecols="B:G"
)

df_p.columns = ["id", "prospect", "salesperson", "close_ratio",
                "goal_cpm", "penalty"]

# Check: 30 prospects, 6 salespeople with 5 prospects each
assert len(df_p) == 30
assert df_p["salesperson"].value_counts().tolist() == [5] * 6


# ------------------------------------------------------------
# D. The real schedule: 'Schedule Release 1-Min Qualifer', rows 13-39
# ------------------------------------------------------------

# One row per actual game, released in August. Games with
# low-interest teams are already cut 30% (one team) or 40% (two).
#
# col C = game ID
# col D = game name
# col F = rate per spot ($)
# col H = original audience per game (000s)
# col I = revised audience after the dud-game cuts (000s)

df_sched = pd.read_excel(
    DATA_FILE,
    sheet_name="Schedule Release 1-Min Qualifer",
    header=None,
    skiprows=12,
    nrows=27,
    usecols="C,D,F,H,I"
)

df_sched.columns = ["game_id", "game", "rate", "original_000s", "revised_000s"]

revised_000s = df_sched["revised_000s"].sum()

# Check against the spreadsheet's own totals (row 40)
assert len(df_sched) == 27
assert df_sched["rate"].sum() == 108_000
assert abs(df_sched["original_000s"].sum() - 1651.38) < 0.001
assert abs(revised_000s - 1465.938) < 0.001


print(f"Package price:  ${package_price:,.0f}")
print(f"Baseline 000s:  {baseline_000s:,.1f}   (July: generic games)")
print(f"Revised 000s:   {revised_000s:,.1f}   (August: dud games cut)")
print()
print(df_scen)
print()
df_p

In [ ]:
# ============================================================
# 1. INDEX SETS
# ============================================================

# customers: the 30 prospects, read from the spreadsheet

customers = df_p["id"].tolist()

# packages: how much audience we guarantee, above the baseline.
# Same 4 names as the base model. The levels match the
# 3 possible Nielsen increases, plus no increase at all.

packages = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]

# discounts: how much we take off the $108,000 price (NEW)

discounts = [0.00, 0.05, 0.10]

# scenarios: the 3 possible Nielsen increases, from the spreadsheet

scenarios = df_scen["label"].tolist()


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = dict(zip(df_p["id"], df_p["goal_cpm"]))

# Makegood multiplier: if we fall short, customer c is owed
# penalty[c] times the value of the missing audience.

underdelivery_penalty = dict(zip(df_p["id"], df_p["penalty"]))


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

# Guarantee level for each package (TEAM ASSUMPTION)

guarantee = {
    "conservative": 0.00,
    "medium": 0.15,
    "premium": 0.20,
    "aggressive": 0.25
}

# Audience we promise the customer (000s)

guaranteed_000s = {}

for p in packages:
    guaranteed_000s[p] = baseline_000s * (1 + guarantee[p])

# Price and CPM now depend on the package AND the discount.
# A bigger guarantee or a bigger discount means a lower CPM.

price = {}
cpm = {}

for p in packages:
    for d in discounts:
        price[p, d] = package_price * (1 - d)
        cpm[p, d] = price[p, d] / guaranteed_000s[p]


# ============================================================
# 4. ACTUAL AUDIENCE OUTCOMES
# ============================================================

# 3 separate outcomes that already add up to 100%.
#
# The station plans for the dud games: the audience it will
# actually deliver starts from the REVISED audience (after the
# dud-game cuts), not the July baseline.
#
# Actual audience = revised x (1 + Nielsen increase)
#
# (Guarantees in section 3 stay on the July baseline, because
#  that's what the customer was promised.)

probability = dict(zip(df_scen["label"], df_scen["probability"]))

actual_000s = {}

for s in scenarios:
    increase = df_scen.loc[df_scen["label"] == s, "increase"].iloc[0]
    actual_000s[s] = revised_000s * (1 + increase)


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,p,d] = 1 if customer c's goal CPM is high enough
# to accept package p at discount d.
#
# Otherwise eligible[c,p,d] = 0.

eligible = {}

for c in customers:
    for p in packages:
        for d in discounts:

            if cpm[p, d] <= goal_cpm[c]:
                eligible[c, p, d] = 1
            else:
                eligible[c, p, d] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

# Shortfall (000s) x CPM the customer paid ($ per 000) = value
# of the missing audience in dollars. The customer is owed that
# value times their penalty multiplier.

expected_penalty = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_penalty[c, p, d] = sum(
                probability[s] *
                max(guaranteed_000s[p] - actual_000s[s], 0) *
                cpm[p, d] *
                underdelivery_penalty[c]
                for s in scenarios
            )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for p in packages:
        for d in discounts:

            expected_profit[c, p, d] = (
                price[p, d]
                - expected_penalty[c, p, d]
            )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Optimization_Schedule")


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,p,d] = 1 if customer c receives package p at discount d
#            0 otherwise

x = model.addVars(
    customers,
    packages,
    discounts,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package
# (one guarantee level at one discount).
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p, d] for p in packages for d in discounts) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, p, d] <= eligible[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have 20 advertising spots, so we can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. OBJECTIVE FUNCTION
# ============================================================

# Maximize expected profit:
#
# Revenue from the package (after discount)
# minus
# Expected underdelivery penalty

model.setObjective(
    gp.quicksum(
        expected_profit[c, p, d] * x[c, p, d]
        for c in customers
        for p in packages
        for d in discounts
    ),
    GRB.MAXIMIZE
)


# ============================================================
# 12. SOLVE MODEL
# ============================================================

model.optimize()


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

print("\nPACKAGE OPTIONS (CPM by guarantee and discount)")
print("=" * 75)

for p in packages:

    line = f"{p.capitalize():12} | Guarantee: +{guarantee[p]:>4.0%} |"

    for d in discounts:

        num_eligible = sum(eligible[c, p, d] for c in customers)

        line += f" {d:>4.0%} off: ${cpm[p, d]:>5.2f} ({num_eligible:>2} eligible) |"

    print(line)

print("\nSHORTFALL BY SCENARIO (000s, after dud-game cuts)")
print("=" * 75)

for s in scenarios:

    line = f"{s:13} | Prob: {probability[s]:>4.0%} | Actual: {actual_000s[s]:>7,.1f} |"

    for p in packages:
        line += f" {p}: {max(guaranteed_000s[p] - actual_000s[s], 0):>5.1f} |"

    print(line)

if model.status == GRB.OPTIMAL:

    print("\nOPTIMAL ADVERTISING PLAN")
    print("=" * 75)

    total_revenue = 0
    total_expected_penalty = 0
    packages_sold = 0

    for c in customers:
        for p in packages:
            for d in discounts:

                if x[c, p, d].X > 0.5:

                    revenue = price[p, d]
                    penalty = expected_penalty[c, p, d]
                    profit = expected_profit[c, p, d]

                    total_revenue += revenue
                    total_expected_penalty += penalty
                    packages_sold += 1

                    print(
                        f"{c}: "
                        f"{p.capitalize():12} | "
                        f"Discount: {d:>4.0%} | "
                        f"Goal CPM: ${goal_cpm[c]:>5.2f} | "
                        f"CPM: ${cpm[p, d]:>5.2f} | "
                        f"Penalty: {underdelivery_penalty[c]}x | "
                        f"Revenue: ${revenue:>9,.0f} | "
                        f"Expected Penalty: ${penalty:>9,.2f} | "
                        f"Expected Profit: ${profit:>10,.2f}"
                    )

    print("=" * 75)

    print(f"Packages Sold:       {packages_sold} of 20")
    print(f"Total Revenue:       ${total_revenue:,.2f}")
    print(f"Expected Penalties:  ${total_expected_penalty:,.2f}")
    print(f"Expected Profit:     ${model.ObjVal:,.2f}")

else:

    print("No optimal solution found.")

### What the results show

**1. The dud games multiply the makegood risk about 6 times.** This is the real-world problem Ed described.

| | Expected penalties | Expected profit |
| --- | ---: | ---: |
| Step 3a plan, as the station expects it in July | \$85,711 | \$1,609,889 |
| Same plan, once the dud games are known | \$513,444 | \$1,182,156 |
| **This step: planned with the duds in mind** | **\$504,307** | **\$1,185,893** |

**2. Planning for the duds barely changes the decisions.** The same 17 customers buy, and only one switches: **C26** moves from premium at full price to medium at 5% off. With the duds, premium at full price nets \$108,000 − \$24,722 expected penalty = \$83,278, while medium at 5% off nets \$102,600 − \$15,589 = \$87,011. Planning ahead improves profit by about \$3,700.

**Why so little change?** With one goal (maximize profit), every sale still pays for itself, even after its expected penalty. So the model still sells to everyone who qualifies. The duds change the *size* of the risk, not *who* to sell to.

**3. This sets up the next step.** Step 3d adds a second goal: keep makegood risk low. With about \$500,000 of risk (instead of \$86,000 without the duds), limiting risk forces real choices: which customers to give up, and who should get a safer guarantee with a bigger discount.

**Worked example (C19, medium at 10% off, 3x penalty):**
- Expected shortfall = 50% × 190.17 + 35% × 116.87 + 15% × 43.57 = **142.53** (000s)
- Value of the missing audience = 142.53 × \$51.81 CPM = **\$7,384**
- Expected penalty = \$7,384 × 3 = **\$22,153**

### The two ways we could have built this step

| | What it does | Upside | Downside |
| --- | --- | --- | --- |
| **Plan for the duds (what we did)** | The model uses the after-duds audience when it decides | The model sees the real risk, so Step 3d's trade-off has real weight | Assumes the station can anticipate the duds in July (stated as a simplification) |
| Plan in July, check in August | Keep Step 3a's model; separately check its plan against the after-duds audience | Matches the real timeline exactly | The model never sees the real risk, so Step 3d would work with only \$86,000 of risk |

We chose the first because the goal of the next step is to manage the real risk, and the model can only manage a risk it can see.

### Changes from Step 3a

1. **Data cell:** new block D reads the 27-game schedule (`df_sched`) and sums the revised audience into `revised_000s`, with checks against the spreadsheet's totals.
2. **Section 4:** `actual_000s[s] = revised_000s * (1 + increase)` (was `baseline_000s`).
3. **Display:** new table showing the shortfall for each package in each scenario.
4. Everything else is unchanged: sets, guarantees, CPMs, eligibility, penalty formula, variables, constraints, and objective.